In [1]:
!pip install langchain langchain-community pypdf unstructured

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 22.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 12.4 MB/s eta 0:00:00
  Create

In [2]:
# --- BLOCK 1: INSTALLATION ---
print("⏳ Installing libraries... (This takes about 45 seconds)")
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters pypdf chromadb sentence-transformers python-pptx unstructured

# --- BLOCK 2: MOUNT DRIVE ---
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Drive already mounted.")

# --- BLOCK 3: CODE EXECUTION ---
import pickle

# Import loaders
try:
    from langchain_community.document_loaders import PyPDFLoader, UnstructuredPowerPointLoader
except ImportError:
    from langchain.document_loaders import PyPDFLoader, UnstructuredPowerPointLoader

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

# CONFIGURATION
BASE_PATH = "/content/drive/MyDrive/GenAiProject_Dataset"

def run_pipeline(base_path, chunk_size=500):
    print(f"\n🚀 STARTING PIPELINE in: {base_path}")

    # 1. Count Files (PDF + PPTX)
    files_to_process = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.lower().endswith((".pdf", ".pptx")):
                files_to_process.append(os.path.join(root, file))

    pdf_count = sum(1 for f in files_to_process if f.lower().endswith(".pdf"))
    pptx_count = sum(1 for f in files_to_process if f.lower().endswith(".pptx"))

    print(f"📊 Found {len(files_to_process)} files: {pdf_count} PDFs, {pptx_count} PPTX")

    if len(files_to_process) == 0:
        print("❌ STOPPING: No PDFs or PPTX found. Please check your folder path.")
        return

    # 2. Load
    all_docs = []
    successful = 0
    failed = 0

    for i, f in enumerate(files_to_process):
        try:
            # Choose loader based on file type
            if f.lower().endswith(".pdf"):
                loader = PyPDFLoader(f)
            else:  # .pptx
                loader = UnstructuredPowerPointLoader(f)

            docs = loader.load()
            all_docs.extend(docs)
            successful += 1

            # Print progress every 5 files
            if (i + 1) % 5 == 0:
                print(f"   Processed {i+1}/{len(files_to_process)} files... (✓ {successful}, ✗ {failed})")
        except Exception as e:
            failed += 1
            print(f"   ⚠️ Error on {os.path.basename(f)}: {e}")

    print(f"\n📈 Loading complete: ✓ {successful} successful, ✗ {failed} failed")

    if len(all_docs) == 0:
        print("❌ STOPPING: No documents were loaded successfully.")
        return

    # 3. Chunk
    print(f"✂️  Chunking {len(all_docs)} pages/slides...")
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=50)
    chunks = splitter.split_documents(all_docs)

    # 4. Save
    save_path = os.path.join(base_path, f"processed_chunks_{chunk_size}.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(chunks, f)

    print(f"\n✅ SUCCESS! Saved {len(chunks)} chunks to:")
    print(f"   {save_path}")
    print(f"\n📊 Final Stats:")
    print(f"   • Total files found: {len(files_to_process)}")
    print(f"   • Successfully processed: {successful}")
    print(f"   • Failed: {failed}")
    print(f"   • Total chunks created: {len(chunks)}")

# Run it
run_pipeline(BASE_PATH, chunk_size=500)

⏳ Installing libraries... (This takes about 45 seconds)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 13.1 MB/s eta 0:00:00
✅ Drive already mounted.

🚀 STARTING PIPELINE in: /content/drive/MyDrive/GenAiProject_Dataset
📊 Found 30 files: 3 PDFs, 27 PPTX
   Processed 5/30 files... (✓ 5, ✗ 0)
   Processed 10/30 files... (✓ 10, ✗ 0)
   Processed 15/30 files... (✓ 15, ✗ 0)
   Processed 20/30 files... (✓ 20, ✗ 0)
   Processed 25/30 files... (✓ 25, ✗ 0)
   Processed 30/3